# Atari Breakout - Double DQN with Prioritized Experience Replay

**Projet de Reinforcement Learning - Albert School**

**Auteurs:** Sara Ben Abdelkader, Saber Dhib

**Date:** 27 octobre 2025

---

## Table des matières
1. Introduction et contexte
2. Description de l'environnement
3. Architecture et algorithme
4. Entraînement et hyperparamètres
5. Résultats et analyse
6. Conclusion

## 1. Introduction et contexte

Ce projet implémente un agent de **Deep Reinforcement Learning** capable d'apprendre à jouer au jeu Atari **Breakout**.

### Objectifs
- Implémenter un algorithme **Double DQN (DDQN)** avec **Prioritized Experience Replay (PER)**
- Optimiser les hyperparamètres pour maximiser les performances
- Analyser l'apprentissage et la stabilité de l'agent

### Choix techniques
- **Framework:** PyTorch
- **Environnement:** Gymnasium (OpenAI Gym) avec ALE (Atari Learning Environment)
- **Architecture:** Dueling DQN pour séparer value et advantage streams
- **Optimisations:** PER, soft target updates, gradient clipping, cosine annealing

In [ ]:
# Imports
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gymnasium as gym
import ale_py

# Imports locaux
sys.path.append('..')
from src.agent_ddqn import DDQNAgent, DuelingDQN
from src.wrappers import make_env
from src.replay_per import PrioritizedReplayBuffer

# Configuration matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'MPS' if torch.backends.mps.is_available() else 'CUDA' if torch.cuda.is_available() else 'CPU'}")

## 2. Description de l'environnement

### Breakout (ALE/Breakout-v5)

**Objectif:** Détruire tous les blocs en haut de l'écran avec une balle, contrôlée par une palette.

**Espace d'observation:**
- 4 frames empilées (84×84 pixels, niveaux de gris)
- Prétraitement: downsampling, grayscale, normalisation

**Espace d'actions:**
- 4 actions discrètes: NOOP, FIRE, RIGHT, LEFT

**Récompenses:**
- +1 pour chaque bloc détruit
- 0 sinon

**Terminaison:**
- Perte de toutes les vies
- Destruction de tous les blocs

In [ ]:
# Créer et visualiser l'environnement
env = make_env('ALE/Breakout-v5', seed=42)
state, info = env.reset()

print(f"Observation shape: {state.shape}")
print(f"Action space: {env.action_space}")
print(f"Action meanings: NOOP=0, FIRE=1, RIGHT=2, LEFT=3")

# Visualiser les 4 frames empilées
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for i in range(4):
    axes[i].imshow(state[i], cmap='gray')
    axes[i].set_title(f'Frame {i+1}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

env.close()

## 3. Architecture et algorithme

### 3.1 Dueling DQN Architecture

Notre réseau utilise une **Dueling architecture** qui sépare:
- **Value stream** V(s): valeur de l'état
- **Advantage stream** A(s,a): avantage de chaque action

$$Q(s,a) = V(s) + \left(A(s,a) - \frac{1}{|\mathcal{A}|} \sum_{a'} A(s,a')\right)$$

**Architecture:**
```
Input (4×84×84)
  ↓
Conv2D(32, 8×8, stride=4) + ReLU
  ↓
Conv2D(64, 4×4, stride=2) + ReLU
  ↓
Conv2D(64, 3×3, stride=1) + ReLU
  ↓
Flatten (3136)
  ↓
  ├─ FC(512) [Value stream] → FC(1) [V(s)]
  └─ FC(512) [Advantage stream] → FC(4) [A(s,a)]
  ↓
Combine → Q(s,a)
```

In [ ]:
# Inspecter le modèle
model = DuelingDQN(in_channels=4, n_actions=4)
print(model)
print(f"\nNombre total de paramètres: {sum(p.numel() for p in model.parameters()):,}")
print(f"Paramètres entraînables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Test forward pass
dummy_input = torch.randn(1, 4, 84, 84)
with torch.no_grad():
    output = model(dummy_input)
print(f"\nOutput shape: {output.shape}")
print(f"Sample Q-values: {output[0]}")

### 3.2 Double DQN avec Prioritized Experience Replay

**Double DQN** résout le problème de surestimation des Q-values:
- Sélection de l'action avec le **réseau online**
- Évaluation avec le **réseau target**

$$y_t = r_t + \gamma Q_{\text{target}}(s_{t+1}, \arg\max_a Q_{\text{online}}(s_{t+1}, a))$$

**Prioritized Experience Replay (PER):**
- Échantillonne les transitions selon leur **TD-error** $\delta$
- Priorité: $p_i = |\delta_i| + \epsilon$
- Importance sampling weights pour corriger le biais

**Autres optimisations:**
- **Soft target update:** $\theta_{\text{target}} \leftarrow \tau \theta_{\text{online}} + (1-\tau) \theta_{\text{target}}$
- **Huber loss:** robuste aux outliers
- **Gradient clipping:** stabilité
- **Cosine annealing:** décroissance du learning rate

## 4. Entraînement et hyperparamètres

### 4.1 Configuration des hyperparamètres

In [ ]:
# Hyperparamètres optimisés
HYPERPARAMS = {
    # Apprentissage
    'learning_rate': 1e-4,
    'gamma': 0.99,
    'batch_size': 32,
    
    # Replay buffer
    'buffer_capacity': 500_000,
    'min_replay_size': 80_000,
    'per_alpha': 0.6,
    'per_beta_start': 0.4,
    'per_beta_frames': 1_000_000,
    
    # Exploration
    'epsilon_start': 1.0,
    'epsilon_final': 0.01,
    'epsilon_decay_frames': 1_500_000,
    
    # Target network
    'target_update_tau': 0.001,
    
    # Training
    'total_frames': 3_000_000,
}

# Afficher sous forme de tableau
df_hp = pd.DataFrame(list(HYPERPARAMS.items()), columns=['Paramètre', 'Valeur'])
print(df_hp.to_string(index=False))

### 4.2 Lancement de l'entraînement

Pour lancer l'entraînement avec la configuration optimisée:

```bash
python -m src.train_improved \
    --env ALE/Breakout-v5 \
    --total-frames 3000000 \
    --save-dir runs_improved \
    --seed 42
```

L'entraînement sauvegarde automatiquement:
- Checkpoints réguliers tous les 50k frames
- Le meilleur modèle (based sur eval)
- Logs d'entraînement dans `training_log.csv`
- Hyperparamètres dans `hyperparams.json`

## 5. Résultats et analyse

### 5.1 Courbes d'apprentissage

In [ ]:
# Charger les logs d'entraînement
# Modifier le chemin selon votre run
log_paths = [
    '../runs/training_log.csv',
    '../runs_improved/training_log.csv'
]

dfs = []
for path in log_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['run'] = os.path.basename(os.path.dirname(path))
        dfs.append(df)
        print(f"✓ Chargé: {path} ({len(df)} lignes)")
    else:
        print(f"⚠ Non trouvé: {path}")

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal: {len(df_all)} lignes")
    print(df_all.head(10))
else:
    print("Aucun log trouvé.")

In [ ]:
# Tracer les courbes d'apprentissage
if dfs:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    for run_name in df_all['run'].unique():
        df_run = df_all[df_all['run'] == run_name]
        
        # 1. Reward par épisode
        df_episodes = df_run[df_run['ep_return'].notna()]
        if len(df_episodes) > 0:
            axes[0, 0].plot(df_episodes['frame'], df_episodes['mean100'], 
                           label=f'{run_name} (mean 100)', alpha=0.8)
            axes[0, 0].scatter(df_episodes['frame'], df_episodes['ep_return'], 
                              alpha=0.1, s=5)
        
        # 2. Epsilon decay
        axes[0, 1].plot(df_run['frame'], df_run['epsilon'], label=run_name, alpha=0.8)
        
        # 3. Loss
        df_loss = df_run[df_run['loss'].notna()]
        if len(df_loss) > 0:
            axes[1, 0].plot(df_loss['frame'], df_loss['loss'], label=run_name, alpha=0.6)
        
        # 4. Eval scores
        df_eval = df_run[df_run['eval_mean'].notna()]
        if len(df_eval) > 0:
            axes[1, 1].errorbar(df_eval['frame'], df_eval['eval_mean'], 
                               yerr=df_eval['eval_std'], label=run_name, 
                               alpha=0.8, capsize=3)
    
    # Configuration des plots
    axes[0, 0].set_xlabel('Frame')
    axes[0, 0].set_ylabel('Episode Return')
    axes[0, 0].set_title('Reward over Time')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    axes[0, 1].set_xlabel('Frame')
    axes[0, 1].set_ylabel('Epsilon')
    axes[0, 1].set_title('Exploration Rate')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    axes[1, 0].set_xlabel('Frame')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].set_title('Training Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    axes[1, 0].set_ylim([0, 2])
    
    axes[1, 1].set_xlabel('Frame')
    axes[1, 1].set_ylabel('Eval Score')
    axes[1, 1].set_title('Evaluation Performance')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('../reports/plots/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✓ Figure sauvegardée: reports/plots/training_curves.png")

### 5.2 Évaluation du modèle final

In [ ]:
# Charger le meilleur checkpoint
checkpoint_path = '../runs_improved/best.pt'  # Ou '../runs/final_dueling.pt'

if not os.path.exists(checkpoint_path):
    checkpoint_path = '../runs/final_dueling.pt'

print(f"Chargement du checkpoint: {checkpoint_path}")

# Créer l'agent
agent = DDQNAgent(n_actions=4)

# Charger le checkpoint
checkpoint = torch.load(checkpoint_path, map_location='cpu')

if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    agent.online.load_state_dict(checkpoint['model_state_dict'])
    print(f"Checkpoint chargé (frame {checkpoint.get('frame', 'N/A')})")
else:
    agent.online.load_state_dict(checkpoint, strict=True)
    print("State dict chargé")

agent.online.eval()
print("✓ Modèle prêt pour l'évaluation")

In [ ]:
# Évaluer sur 30 épisodes
from tqdm.notebook import tqdm

env = make_env('ALE/Breakout-v5', seed=42)
n_episodes = 30
epsilon = 0.01

scores = []
episode_lengths = []

agent.eps_final = epsilon

for ep in tqdm(range(n_episodes), desc="Évaluation"):
    state, _ = env.reset()
    done = False
    total_reward = 0.0
    steps = 0
    
    while not done:
        action = agent.act(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1
        state = next_state
    
    scores.append(total_reward)
    episode_lengths.append(steps)

env.close()

# Statistiques
print("\n" + "="*60)
print("📊 RÉSULTATS D'ÉVALUATION")
print("="*60)
print(f"Score moyen:      {np.mean(scores):.2f} ± {np.std(scores):.2f}")
print(f"Score médian:     {np.median(scores):.2f}")
print(f"Score min/max:    {np.min(scores):.1f} / {np.max(scores):.1f}")
print(f"Quartiles (25/75): {np.percentile(scores, 25):.2f} / {np.percentile(scores, 75):.2f}")
print(f"Longueur moyenne: {np.mean(episode_lengths):.1f} steps")
print("="*60)

In [ ]:
# Visualiser la distribution des scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(scores, bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(scores), color='red', linestyle='--', label=f'Mean: {np.mean(scores):.1f}')
axes[0].axvline(np.median(scores), color='green', linestyle='--', label=f'Median: {np.median(scores):.1f}')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution des scores')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Score par épisode
axes[1].plot(scores, marker='o', linestyle='-', alpha=0.6)
axes[1].axhline(np.mean(scores), color='red', linestyle='--', label=f'Mean: {np.mean(scores):.1f}')
axes[1].fill_between(range(len(scores)), 
                     np.mean(scores) - np.std(scores), 
                     np.mean(scores) + np.std(scores), 
                     alpha=0.2, color='red')
axes[1].set_xlabel('Épisode')
axes[1].set_ylabel('Score')
axes[1].set_title('Score par épisode')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/plots/evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figure sauvegardée: reports/plots/evaluation_results.png")

### 5.3 Analyse des résultats

**Observations:**

1. **Apprentissage progressif:**
   - L'agent commence avec une performance aléatoire (~0-1 points)
   - Amélioration graduelle vers 8-12 points après 500k frames
   - Convergence vers une politique stable

2. **Exploration vs Exploitation:**
   - Epsilon décroît exponentiellement de 1.0 à 0.01
   - Balance exploration initiale et exploitation finale
   - Permet d'éviter les minima locaux

3. **Stabilité:**
   - Double DQN réduit l'overestimation des Q-values
   - PER améliore l'efficacité d'apprentissage
   - Soft target updates assurent la stabilité

4. **Comparaison avec baseline:**
   - Performance humaine débutante: ~5-15 points
   - Performance humaine experte: ~30-50 points
   - Notre agent: ~8-12 points (comparable à un débutant)

**Limitations identifiées:**
- Temps d'entraînement limité (500k-3M frames)
- Difficulté à apprendre des stratégies complexes (tunneling)
- Variabilité inter-épisodes encore élevée

### 5.4 Améliorations possibles

**Court terme:**
1. ✓ Augmenter le buffer size (200k → 500k)
2. ✓ Utiliser Dueling architecture
3. ✓ Soft target updates au lieu de hard updates
4. ⏳ Entraînement plus long (3M+ frames)
5. ⏳ Curriculum learning

**Long terme:**
1. Rainbow DQN (combiner toutes les améliorations)
2. Distributional RL (C51, QR-DQN)
3. Noisy Networks pour exploration
4. Multi-step returns
5. Ensemble methods

## 6. Conclusion

### Résumé des accomplissements

Dans ce projet, nous avons:

1. **Implémenté** un agent Double DQN avec Dueling architecture et PER
2. **Optimisé** les hyperparamètres pour améliorer les performances
3. **Entraîné** l'agent sur l'environnement Atari Breakout
4. **Analysé** les résultats et identifié les points d'amélioration
5. **Développé** des outils d'évaluation et de visualisation

### Performance finale

- **Score moyen:** 8-12 points par épisode
- **Stabilité:** Convergence stable après ~500k frames
- **Comportement:** L'agent apprend à suivre la balle et positionner la palette

### Contributions individuelles

**Sara Ben Abdelkader:**
- Implementation de l'architecture DDQN
- Développement des wrappers et preprocessing
- Scripts d'entraînement et visualisation
- Gestion du pipeline d'évaluation

**Saber Dhib:**
- Tuning des hyperparamètres
- Analyse des résultats
- Rédaction du rapport
- Comparaisons avec baselines

### Perspectives

Ce projet démontre l'efficacité du Deep Reinforcement Learning pour apprendre des politiques complexes à partir de pixels bruts. Les techniques modernes (Double DQN, Dueling, PER) permettent un apprentissage stable et efficace.

L'extension vers Rainbow DQN ou des méthodes policy-gradient (PPO, SAC) pourrait améliorer significativement les performances.

---

**Merci !**

## Annexe: Commandes utiles

```bash
# Installation des dépendances
pip install -r requirements.txt

# Entraînement
python -m src.train_improved --total-frames 3000000 --save-dir runs_improved

# Évaluation headless
python -m src.eval --model-path runs_improved/best.pt --n-episodes 30 --output results.csv

# Visualisation de l'agent
python -m src.watch_agent --model-path runs_improved/best.pt --episodes 3

# Conversion de checkpoint
python -m src.convert_checkpoint --input runs/old.pt --output runs/new_dueling.pt

# Tests unitaires
python tests/test_suite.py
```